In [4]:
import pandas as pd

crimes = pd.read_csv("data/original/Crimes_-_2001_to_Present_20261104.csv")
crime_dates = pd.to_datetime(crimes["Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
cleaned_crimes = crimes.loc[crime_dates.dt.year.eq(2025), ["Date", "Primary Type"]].copy()
cleaned_crimes.to_csv("data/processed/crimes_cleaned.csv", index=False)

cleaned_crimes.head()

,Date,Primary Type
0,12/31/2025 11:54:00 PM,BATTERY
1,12/31/2025 11:44:00 PM,BATTERY
2,12/31/2025 11:25:00 PM,BATTERY
3,12/31/2025 10:57:00 PM,ASSAULT
4,12/31/2025 10:43:00 PM,BATTERY


In [5]:
import pandas as pd

crimes = pd.read_csv("data/processed/crimes_cleaned.csv")
weather = pd.read_csv("data/processed/Chicago_midway_weather_cleaned.csv")

crimes["Date"] = pd.to_datetime(
    crimes["Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"
).dt.normalize()
crimes = crimes[crimes["Date"].dt.year.eq(2025)]

# Build full daily dates from the cleaned 2025 weather month/day columns.
weather["Date"] = pd.to_datetime(
    {"year": 2025, "month": weather["MONTH"], "day": weather["DAY"]}
)
weather = weather.rename(columns={"TAVG": "Temperature", "PRCP": "Precipitation"})
weather["Temperature"] = pd.to_numeric(weather["Temperature"], errors="coerce")
weather["Precipitation"] = pd.to_numeric(weather["Precipitation"], errors="coerce")
daily_weather = weather[["Date", "Temperature", "Precipitation"]]

# Count assault and battery incidents separately for each day.
assault_battery = crimes[crimes["Primary Type"].isin(["ASSAULT", "BATTERY"])]
daily_crimes = (
    assault_battery.dropna(subset=["Date"])
    .groupby(["Date", "Primary Type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["ASSAULT", "BATTERY"], fill_value=0)
    .rename(columns={"ASSAULT": "Assault Count", "BATTERY": "Battery Count"})
    .reset_index()
)

# Keep all calendar days, including days with no reported incidents.
date_range = pd.date_range("2025-01-01", "2025-12-31", freq="D", name="Date")
daily_crimes_weather = (
    pd.DataFrame({"Date": date_range})
    .merge(daily_crimes, on="Date", how="left")
    .merge(daily_weather, on="Date", how="left", validate="one_to_one")
)
count_columns = ["Assault Count", "Battery Count"]
daily_crimes_weather[count_columns] = daily_crimes_weather[count_columns].fillna(0).astype(int)
daily_crimes_weather.to_csv("data/processed/daily_crimes_weather.csv", index=False)

print("Analysis period:", daily_crimes_weather["Date"].min().date(), "to", daily_crimes_weather["Date"].max().date())
daily_crimes_weather.head()

Analysis period: 2025-01-01 to 2025-12-31


,Date,Assault Count,Battery Count,Temperature,Precipitation
0,2025-01-01,5,35,30.5,0.00
1,2025-01-02,8,21,29.0,0.00
2,2025-01-03,2,14,23.0,0.00
3,2025-01-04,3,20,20.0,0.00
4,2025-01-05,11,22,23.5,0.01
